In [1]:
!pip install -r requirements.txt

In [2]:
# pip install --upgrade transformers accelerate

In [3]:
import pathlib as pl
import requests
import zipfile
from corus import load_ne5, load_lenta
import re
from datasets import Dataset, Sequence, ClassLabel
from transformers import (
    AutoTokenizer, AutoModelForTokenClassification,
    AutoModelForMaskedLM,
    TrainingArguments, Trainer,
    DataCollatorForTokenClassification,
    DataCollatorForLanguageModeling
)
import evaluate
import numpy as np
import tabulate
import torch

/home/semyon/python_projects/NLP/nlp-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print(device)

cpu


In [5]:
url = "http://www.labinform.ru/pub/named_entities/collection5.zip"

path = "Collection5.zip"
zip_path = pl.Path.cwd()/path

if not pl.Path.exists(zip_path):
    response = requests.get(url, stream=True)
    if response.status_code == 200:
        with open(path, "wb") as file:
            for chunk in response.iter_content(chunk_size=8192):
                file.write(chunk)
        print("Файл успешно загружен.")
    else:
        print("Ошибка при загрузке файла")
else:
    print("Файл уже существует")

Файл уже существует


In [6]:
extract_dir = pl.Path.cwd()
if pl.Path.exists(zip_path):
    with zipfile.ZipFile(path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)

    if not pl.Path.exists(extract_dir):
        pl.Path.mkdir(extract_dir, exist_ok=True)

    print("Архив распакован")
else:
    print("Файл не существует.")

Архив распакован


In [7]:
dataset = load_ne5("Collection5")

In [8]:
next(dataset)

Ne5Markup(
    id='479',
    text='С.Собянин назначил глав управ районов трех округов Москвы.\r\n\r\nМэр Москвы Сергей Собянин назначил глав управ районов трех округов Москвы. Как сообщили РБК в правительстве столицы, соответствующие распоряжения уже подписаны. Назначены главы управ в Северном, Западном и Южном административных округах города.\r\n\r\nВ Северном округе назначены главы управ 8 районов. Управу района Сокол возглавил Виталий Аксенов, Дмитровского района - Владимир Назаров, Западного Дегунино - Сергей Овчинников, Савеловского - Станислав Одиноков, Тимирязевского - Владимир Палкин, Коптево - Владимир Перов, Войковского - Сергей Сидоров, Левобережного - Виктор Ярцев.\r\n\r\nВ Западном округе назначены 13 глав управ районов. Район Внуково возглавил Павел Авеков, Солнцево - Константин Бусыгин, Филевский парк - Антон Гудзь, Кунцево - Назим Намазов, Крылатское - Виталий Никитин, Очаково-Матвеевское - Сергей Новиков, Раменки - Игорь Окунев, Можайский - Михаил Решетников, Проспект 

In [9]:
def convert_markup_to_ner(item):
    text = item.text
    tokens = []
    offsets = []
    for match in re.finditer(r"\S+", text):
        tokens.append(match.group())
        offsets.append((match.start(), match.end()))

    labels = ["O"] * len(tokens)

    for span in item.spans:
        token_indices = [
            i for i, (tstart, tend) in enumerate(offsets) if not (tend <= span.start or tstart >= span.stop)
        ]
        if token_indices:
            labels[token_indices[0]] = "B-" + span.type
            for idx in token_indices[1:]:
                labels[idx] = "I-" + span.type

    return {"id": item.id, "tokens": tokens, "ner_tags": labels}

In [10]:
data_list = [convert_markup_to_ner(item) for item in dataset]
ner_dataset = Dataset.from_list(data_list)

In [11]:
unique_labels = set()
for example in ner_dataset:
    unique_labels.update(example["ner_tags"])

unique_labels = sorted(list(unique_labels))
label_to_id = {label: i for i, label in enumerate(unique_labels)}

In [12]:
label_to_id

{'B-GEOPOLIT': 0,
 'B-LOC': 1,
 'B-MEDIA': 2,
 'B-ORG': 3,
 'B-PER': 4,
 'I-GEOPOLIT': 5,
 'I-LOC': 6,
 'I-MEDIA': 7,
 'I-ORG': 8,
 'I-PER': 9,
 'O': 10}

In [13]:
def convert_labels(example):
    example["ner_tags"] = [label_to_id[label] for label in example["ner_tags"]]
    return example

In [14]:
ner_dataset = ner_dataset.map(convert_labels)

Map: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████| 999/999 [00:00<00:00, 6571.26 examples/s]


In [15]:
features = ner_dataset.features.copy()
features["ner_tags"] = Sequence(ClassLabel(names=unique_labels))
ner_dataset = ner_dataset.cast(features)

Casting the dataset: 100%|██████████████████████████████████████████████████████████████████████████████████| 999/999 [00:00<00:00, 193789.18 examples/s]


In [16]:
print(ner_dataset[0])

{'id': '15_01_13b', 'tokens': ['А.Хинштейн:', 'Руководство', '"Военторга"', 'попалось', 'на', 'продовольственной', 'афере', 'А.Хинштейн:', 'Руководство', '"Военторга"', 'попалось', 'на', 'продовольственной', 'афере', 'Руководство', 'ОАО', '"Военторг"', 'подозревается', 'в', 'злоупотреблении', 'полномочиями', 'при', 'поставках', 'продовольствия', 'в', 'армию.', 'Об', 'этом', 'сообщил', 'РБК', 'депутат', 'Государственной', 'думы', 'Александр', 'Хинштейн,', 'ссылаясь', 'на', 'ответ,', 'полученный', 'из', 'Главной', 'военной', 'прокуратуры', '(ГВП).', 'По', 'словам', 'депутата,', 'основные', 'претензии', 'у', 'правоохранительных', 'органов', 'к', 'теперь', 'уже', 'бывшей', 'главе', '"Военторга"', 'Марине', 'Лопатиной,', 'которая', 'лишилась', 'своего', 'поста', 'в', 'связи', 'с', 'расследованием', 'этого', 'дела.', 'Депутат', 'уточнил,', 'что', 'гендиректор', 'М.Лопатина', 'снята', 'с', 'должности', 'по', 'прокурорскому', 'представлению.', 'При', 'этом', 'А.Хинштейн', 'отметил,', 'что', 'к

In [17]:
tokenizer = AutoTokenizer.from_pretrained("cointegrated/rubert-tiny2")

In [18]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=128,
        padding="max_length"
    )
    all_labels = []
    for i, labels in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(labels[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        all_labels.append(label_ids)
    tokenized_inputs["labels"] = all_labels
    return tokenized_inputs

In [19]:
tokenized_dataset = ner_dataset.map(tokenize_and_align_labels, batched=True)

Map: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████| 999/999 [00:00<00:00, 2433.63 examples/s]


In [20]:
tokenized_dataset = tokenized_dataset.train_test_split(test_size=0.2)

In [21]:
train_dataset = tokenized_dataset["train"]
test_dataset = tokenized_dataset["test"]

In [22]:
num_labels = len(unique_labels)
model = AutoModelForTokenClassification.from_pretrained("cointegrated/rubert-tiny2", num_labels=num_labels)

Some weights of BertForTokenClassification were not initialized from the model checkpoint at cointegrated/rubert-tiny2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [23]:
training_args = TrainingArguments(
    output_dir="models/ner_results",
    eval_strategy="steps",
    save_strategy="steps",
    save_steps=1000,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    max_steps=10000,
    logging_steps=500,
    weight_decay=0.01,
    logging_dir="./logs",
    report_to="none",
    load_best_model_at_end=True,
    save_total_limit=2,
)

In [24]:
metric = evaluate.load("seqeval")

In [25]:
label_list = tokenized_dataset['train'].features['ner_tags'].feature.names
print("Label list:", label_list)

Label list: ['B-GEOPOLIT', 'B-LOC', 'B-MEDIA', 'B-ORG', 'B-PER', 'I-GEOPOLIT', 'I-LOC', 'I-MEDIA', 'I-ORG', 'I-PER', 'O']


In [26]:
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [27]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=DataCollatorForTokenClassification(tokenizer=tokenizer),
    compute_metrics=compute_metrics
)


In [28]:
print("Метрики ДО дообучения:")
pre_training_results = trainer.evaluate()
print(tabulate.tabulate(
    pre_training_results.items(),
    headers=["Метрика", "Значение"],
    tablefmt="grid",
    floatfmt=".4f"
))

Метрики ДО дообучения:


/home/semyon/python_projects/NLP/nlp-env/lib/python3.12/site-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


+-----------------------------+------------+
| Метрика                     |   Значение |
+=============================+============+
| eval_loss                   |     2.4678 |
+-----------------------------+------------+
| eval_model_preparation_time |     0.0018 |
+-----------------------------+------------+
| eval_precision              |     0.0151 |
+-----------------------------+------------+
| eval_recall                 |     0.0817 |
+-----------------------------+------------+
| eval_f1                     |     0.0255 |
+-----------------------------+------------+
| eval_accuracy               |     0.0549 |
+-----------------------------+------------+
| eval_runtime                |     0.7940 |
+-----------------------------+------------+
| eval_samples_per_second     |   251.8780 |
+-----------------------------+------------+
| eval_steps_per_second       |    16.3720 |
+-----------------------------+------------+


In [29]:
trainer.train()

Step,Training Loss,Validation Loss,Model Preparation Time,Precision,Recall,F1,Accuracy
500,0.420800,0.139603,0.001800,0.821612,0.876514,0.848175,0.964820
1000,0.069900,0.108269,0.001800,0.854344,0.903087,0.878040,0.971059
1500,0.032100,0.104055,0.001800,0.869922,0.912075,0.890500,0.974352
2000,0.018000,0.104934,0.001800,0.880435,0.917937,0.898795,0.976027
2500,0.011500,0.106493,0.001800,0.879581,0.919109,0.898911,0.975911
3000,0.008000,0.109119,0.001800,0.883502,0.918718,0.900766,0.976489
3500,0.005900,0.113385,0.001800,0.880597,0.922235,0.900935,0.976547
4000,0.004400,0.119451,0.001800,0.880491,0.924189,0.901811,0.976142
4500,0.003500,0.119403,0.001800,0.888015,0.926534,0.906866,0.977240
5000,0.002700,0.120712,0.001800,0.885468,0.921454,0.903102,0.976778


/home/semyon/python_projects/NLP/nlp-env/lib/python3.12/site-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/home/semyon/python_projects/NLP/nlp-env/lib/python3.12/site-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/home/semyon/python_projects/NLP/nlp-env/lib/python3.12/site-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/home/semyon/python_projects/NLP/nlp-env/lib/python3.12/site-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(wa

TrainOutput(global_step=10000, training_loss=0.029518825206160546, metrics={'train_runtime': 2734.5931, 'train_samples_per_second': 58.51, 'train_steps_per_second': 3.657, 'total_flos': 282960319641600.0, 'train_loss': 0.029518825206160546, 'epoch': 200.0})

In [30]:
print("Метрики ПОСЛЕ дообучения:")
post_training_results = trainer.evaluate()
print(tabulate.tabulate(
    post_training_results.items(),
    headers=["Метрика", "Значение"],
    tablefmt="grid",
    floatfmt=".4f"
))

Метрики ПОСЛЕ дообучения:


/home/semyon/python_projects/NLP/nlp-env/lib/python3.12/site-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


+-----------------------------+------------+
| Метрика                     |   Значение |
+=============================+============+
| eval_loss                   |     0.1083 |
+-----------------------------+------------+
| eval_model_preparation_time |     0.0018 |
+-----------------------------+------------+
| eval_precision              |     0.8543 |
+-----------------------------+------------+
| eval_recall                 |     0.9031 |
+-----------------------------+------------+
| eval_f1                     |     0.8780 |
+-----------------------------+------------+
| eval_accuracy               |     0.9711 |
+-----------------------------+------------+
| eval_runtime                |     0.5624 |
+-----------------------------+------------+
| eval_samples_per_second     |   355.6420 |
+-----------------------------+------------+
| eval_steps_per_second       |    23.1170 |
+-----------------------------+------------+
| epoch                       |   200.0000 |
+---------

# Попробуйте улучшить качество модели

### Дообучение в MLM режиме на train части, а затем дообучение на NER

In [40]:
train_mlm_dataset = tokenized_dataset['train'].map(
    lambda example: {'text': ' '.join(example['tokens'])},
    remove_columns=tokenized_dataset['train'].column_names  # Удаляем все оригинальные колонки
)

Map: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████| 799/799 [00:00<00:00, 5101.08 examples/s]


In [41]:
train_mlm_dataset

Dataset({
    features: ['text'],
    num_rows: 799
})

In [42]:
mask_tokenizer = AutoTokenizer.from_pretrained("cointegrated/rubert-tiny2")

def tokenize_mlm(examples):
    result = mask_tokenizer(
        examples['text'],
        truncation=True,
        padding='max_length',
        max_length=256,
        return_special_tokens_mask=True
    )
    return result

tokenized_mlm_dataset = train_mlm_dataset.map(
    tokenize_mlm,
    batched=True,
    remove_columns=['text']
)

Map: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████| 799/799 [00:00<00:00, 5563.08 examples/s]


In [43]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=mask_tokenizer,
    mlm=True,
    mlm_probability=0.15
)

In [51]:
mlm_model = AutoModelForMaskedLM.from_pretrained("cointegrated/rubert-tiny2")

training_mlm_args = TrainingArguments(
    output_dir="models/mlm_results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    learning_rate=5e-5,
    weight_decay=0.01,
    logging_steps=500,
    save_steps=1000,
    save_total_limit=2,
    prediction_loss_only=True,
    remove_unused_columns=False,
    logging_dir="./logs_mlm",
    report_to="none",
)

mlm_trainer = Trainer(
    model=mlm_model,
    args=training_mlm_args,
    train_dataset=tokenized_mlm_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator,
)

In [45]:
mlm_trainer.train()

Step,Training Loss


TrainOutput(global_step=150, training_loss=3.145899658203125, metrics={'train_runtime': 556.829, 'train_samples_per_second': 4.305, 'train_steps_per_second': 0.269, 'total_flos': 9146616956928.0, 'train_loss': 3.145899658203125, 'epoch': 3.0})

In [55]:
mlm_trainer.save_model("models/mlm_finetuned_model")

In [56]:
mlm_nle_model = AutoModelForTokenClassification.from_pretrained(
    "models/mlm_finetuned_model", num_labels=num_labels,  # количество ваших NER классов
    # ignore_mismatched_sizes=True  # важно! меняем голову модели
)

Some weights of BertForTokenClassification were not initialized from the model checkpoint at models/mlm_finetuned_model and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [57]:
training_args = TrainingArguments(
    output_dir="models/mlm_ner_results",
    eval_strategy="steps",
    save_strategy="steps",
    save_steps=1000,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    max_steps=10000,
    logging_steps=500,
    weight_decay=0.01,
    logging_dir="./logs",
    report_to="none",
    load_best_model_at_end=True,
    save_total_limit=2,
)

In [60]:
mlm_nle_trainer = Trainer(
    model=mlm_nle_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=DataCollatorForTokenClassification(tokenizer=tokenizer),
    compute_metrics=compute_metrics
)


In [61]:
print("Метрики после дообучения в MLM режиме:")
mlm_training_results = mlm_nle_trainer.evaluate()
print(tabulate.tabulate(
    mlm_training_results.items(),
    headers=["Метрика", "Значение"],
    tablefmt="grid",
    floatfmt=".4f"
))

Метрики после дообучения в MLM режиме:


+-----------------------------+------------+
| Метрика                     |   Значение |
+=============================+============+
| eval_loss                   |     2.4774 |
+-----------------------------+------------+
| eval_model_preparation_time |     0.0012 |
+-----------------------------+------------+
| eval_precision              |     0.0184 |
+-----------------------------+------------+
| eval_recall                 |     0.1059 |
+-----------------------------+------------+
| eval_f1                     |     0.0313 |
+-----------------------------+------------+
| eval_accuracy               |     0.0610 |
+-----------------------------+------------+
| eval_runtime                |     0.6928 |
+-----------------------------+------------+
| eval_samples_per_second     |   288.6670 |
+-----------------------------+------------+
| eval_steps_per_second       |    18.7630 |
+-----------------------------+------------+


In [62]:
mlm_nle_trainer.train()

Step,Training Loss,Validation Loss,Model Preparation Time,Precision,Recall,F1,Accuracy
500,0.408800,0.142183,0.001200,0.821429,0.871825,0.845877,0.964531
1000,0.072000,0.112204,0.001200,0.854228,0.899961,0.876499,0.970597
1500,0.034600,0.109222,0.001200,0.858732,0.905041,0.881279,0.971290
2000,0.019600,0.112734,0.001200,0.873830,0.912075,0.892543,0.972965
2500,0.012700,0.113015,0.001200,0.881177,0.912857,0.896737,0.974814
3000,0.008900,0.115470,0.001200,0.880979,0.914029,0.897200,0.974929
3500,0.006400,0.118290,0.001200,0.881887,0.913247,0.897293,0.976027
4000,0.004800,0.121092,0.001200,0.879505,0.915592,0.897186,0.975391
4500,0.004000,0.125325,0.001200,0.883528,0.915983,0.899463,0.975796
5000,0.003200,0.127201,0.001200,0.884195,0.915983,0.899808,0.975622


/home/semyon/python_projects/NLP/nlp-env/lib/python3.12/site-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/home/semyon/python_projects/NLP/nlp-env/lib/python3.12/site-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/home/semyon/python_projects/NLP/nlp-env/lib/python3.12/site-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/home/semyon/python_projects/NLP/nlp-env/lib/python3.12/site-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(wa

TrainOutput(global_step=10000, training_loss=0.029506268256902696, metrics={'train_runtime': 2428.9375, 'train_samples_per_second': 65.872, 'train_steps_per_second': 4.117, 'total_flos': 282960319641600.0, 'train_loss': 0.029506268256902696, 'epoch': 200.0})

In [63]:
print("Метрики после дообучения на NER модели, дообученной на MLM режиме:")
mlm_training_results = mlm_nle_trainer.evaluate()
print(tabulate.tabulate(
    mlm_training_results.items(),
    headers=["Метрика", "Значение"],
    tablefmt="grid",
    floatfmt=".4f"
))

Метрики после дообучения на NER модели, дообученной на MLM режиме:


/home/semyon/python_projects/NLP/nlp-env/lib/python3.12/site-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


+-----------------------------+------------+
| Метрика                     |   Значение |
+=============================+============+
| eval_loss                   |     0.1122 |
+-----------------------------+------------+
| eval_model_preparation_time |     0.0012 |
+-----------------------------+------------+
| eval_precision              |     0.8542 |
+-----------------------------+------------+
| eval_recall                 |     0.9000 |
+-----------------------------+------------+
| eval_f1                     |     0.8765 |
+-----------------------------+------------+
| eval_accuracy               |     0.9706 |
+-----------------------------+------------+
| eval_runtime                |     0.6746 |
+-----------------------------+------------+
| eval_samples_per_second     |   296.4670 |
+-----------------------------+------------+
| eval_steps_per_second       |    19.2700 |
+-----------------------------+------------+
| epoch                       |   200.0000 |
+---------

### С синтетическими данными

In [ ]:
if not os.path.exists('lenta-ru-news.csv.gz'):
    !wget https://github.com/yutkin/Lenta.Ru-News-Dataset/releases/download/v1.0/lenta-ru-news.csv.gz